In [1]:
import os, json, shutil
from pathlib import Path
import numpy as np
import nibabel as nib
from tqdm import tqdm
import torch

# Fontes RAW oficiais (apenas dados com ground truth)
TRAIN_RAW = Path("/home/henrique.grande/brats_goat_experiments/data/raw/MICCAI2024-BraTS-GoAT-TrainingData-With-GroundTruth")
VAL_RAW   = Path("/home/henrique.grande/brats_goat_experiments/data/raw/MICCAI2024-BraTS-GoAT-ValidationData")

# Base ISOLADA deste experimento (nao sobrescreve outros)
NNUNET_BASE = Path("/home/henrique.grande/brats_goat_experiments/models_configs/nnUnet_base_1fold_supervised_less_val")
NNUNET_RAW = NNUNET_BASE / "nnUNet_raw"
NNUNET_PREP = NNUNET_BASE / "nnUNet_preprocessed"
NNUNET_RES = NNUNET_BASE / "nnUNet_results"
for p in [NNUNET_RAW, NNUNET_PREP, NNUNET_RES]:
    p.mkdir(parents=True, exist_ok=True)

os.environ["nnUNet_raw"] = str(NNUNET_RAW)
os.environ["nnUNet_preprocessed"] = str(NNUNET_PREP)
os.environ["nnUNet_results"] = str(NNUNET_RES)

DATASET_ID = 952
DATASET_NAME = "BraTS-GoAT-base-1fold-supervised-less-val"
DATASET_FOLDER = NNUNET_RAW / f"Dataset{DATASET_ID:03d}_{DATASET_NAME}"
imagesTr = DATASET_FOLDER / "imagesTr"
labelsTr = DATASET_FOLDER / "labelsTr"
imagesVs = DATASET_FOLDER / "imagesVs"
for p in [imagesTr, labelsTr, imagesVs]:
    p.mkdir(parents=True, exist_ok=True)

# Predicoes isoladas (validacao oficial)
PRED_VAL_OFC = Path("/home/henrique.grande/brats_goat_experiments/predictions/nnUnet_base_1fold_supervised_less_val/ofc_validation")
PRED_VAL_OFC.mkdir(parents=True, exist_ok=True)

# Pipeline padrao nnU-Net - configuracao Large (L)
PLANNER    = "nnUNetPlannerResEncL"
PLANS_NAME = "nnUNetResEncUNetLPlans"
TRAINER    = "nnUNetTrainer"   # 1000 epocas (padrao)
CONFIG     = "3d_fullres"
FOLD       = 0

DATASET_FOLDER

PosixPath('/home/henrique.grande/brats_goat_experiments/models_configs/nnUnet_base_1fold_supervised_less_val/nnUNet_raw/Dataset952_BraTS-GoAT-base-1fold-supervised-less-val')

In [2]:
print("=" * 60)
print("  Informacoes da GPU")
print("=" * 60)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("  CUDA   :", torch.version.cuda, "| PyTorch:", torch.__version__)
    print("  GPU    :", props.name)
    print("  VRAM   :", f"{props.total_memory/1e9:.2f} GB")
else:
    print("  CUDA   : NAO")
print("=" * 60)

  Informacoes da GPU
  CUDA   : 12.1 | PyTorch: 2.5.1+cu121
  GPU    : NVIDIA H100 80GB HBM3
  VRAM   : 85.02 GB


In [3]:
def get_patient_ids(directory):
    if not directory.exists():
        return []
    return sorted([d.name for d in directory.iterdir() if d.is_dir()])

train_ids = get_patient_ids(TRAIN_RAW)     # rotulados oficiais (com GT)
val_ofc_ids = get_patient_ids(VAL_RAW)     # validacao oficial (sem GT, so inferencia)

print(f"Treino rotulado (With-GroundTruth): {len(train_ids)} | existe: {TRAIN_RAW.exists()}")
print(f"Validacao oficial:                  {len(val_ofc_ids)} | existe: {VAL_RAW.exists()}")

Treino rotulado (With-GroundTruth): 1351 | existe: True
Validacao oficial:                  451 | existe: True


In [4]:
MODS = ["t1n", "t1c", "t2w", "t2f"]
FILE_ENDING = ".nii.gz"

def case_dir(split_name, case_id):
    if split_name == "train":
        return TRAIN_RAW / case_id
    elif split_name == "val_ofc":
        return VAL_RAW / case_id
    return None

def find_file(folder, key):
    for cand in [folder / f"{key}.nii.gz", folder / f"{key}.nii"]:
        if cand.exists():
            return cand
    cands = sorted(list(folder.glob(f"*{key}*.nii*")))
    if key == "t1n":
        cands = [c for c in cands if "t1c" not in c.name.lower()]
    if not cands:
        raise FileNotFoundError(f"{key} not found in {folder}")
    return cands[0]

def save_as_niigz(src_path, dst_path):
    if src_path.name.lower().endswith(".nii.gz"):
        shutil.copy2(src_path, dst_path)
        return
    nib.save(nib.load(str(src_path)), str(dst_path))

def save_label_niigz(seg_path, dst_path):
    img = nib.load(str(seg_path))
    data = np.asanyarray(img.dataobj).astype(np.int16)
    nib.save(nib.Nifti1Image(data, img.affine, img.header), str(dst_path))

In [5]:
def export_case(split_name, case_id, to_images, to_labels):
    d = case_dir(split_name, case_id)
    for ch, mod in enumerate(MODS):
        save_as_niigz(find_file(d, mod), to_images / f"{case_id}_{ch:04d}{FILE_ENDING}")
    if to_labels is not None:
        save_label_niigz(find_file(d, "seg"), to_labels / f"{case_id}{FILE_ENDING}")

for cid in tqdm(train_ids, desc="Export treino real -> imagesTr/labelsTr"):
    export_case("train", cid, imagesTr, labelsTr)

for cid in tqdm(val_ofc_ids, desc="Export validacao oficial -> imagesVs"):
    export_case("val_ofc", cid, imagesVs, None)

print(f"imagesTr: {len(list(imagesTr.glob(f'*{FILE_ENDING}')))} | labelsTr: {len(list(labelsTr.glob(f'*{FILE_ENDING}')))}")
print(f"imagesVs: {len(list(imagesVs.glob(f'*{FILE_ENDING}')))}")

Export treino real -> imagesTr/labelsTr:   0%|          | 0/1351 [00:00<?, ?it/s]

Export treino real -> imagesTr/labelsTr:   0%|          | 1/1351 [00:00<04:21,  5.17it/s]

Export treino real -> imagesTr/labelsTr:   0%|          | 2/1351 [00:00<04:01,  5.58it/s]

Export treino real -> imagesTr/labelsTr:   0%|          | 3/1351 [00:00<03:45,  5.98it/s]

Export treino real -> imagesTr/labelsTr:   0%|          | 4/1351 [00:00<03:37,  6.20it/s]

...

Export validacao oficial -> imagesVs:  98%|█████████▊| 443/451 [00:15<00:00, 23.67it/s]

Export validacao oficial -> imagesVs:  99%|█████████▉| 446/451 [00:15<00:00, 24.27it/s]

Export validacao oficial -> imagesVs: 100%|█████████▉| 449/451 [00:15<00:00, 24.45it/s]

Export validacao oficial -> imagesVs: 100%|██████████| 451/451 [00:16<00:00, 28.15it/s]

imagesTr: 5404 | labelsTr: 1351
imagesVs: 1804


In [6]:
dataset_json = {
    "channel_names": {"0": "T1n", "1": "T1c", "2": "T2w", "3": "T2f"},
    "labels": {"background": 0, "NCR/NET": 1, "ED": 2, "ET": 3},
    "numTraining": len(list(labelsTr.glob(f"*{FILE_ENDING}"))),
    "file_ending": FILE_ENDING,
}
with open(DATASET_FOLDER / "dataset.json", "w", encoding="utf-8") as f:
    json.dump(dataset_json, f, indent=2, ensure_ascii=False)
DATASET_FOLDER / "dataset.json"

PosixPath('/home/henrique.grande/brats_goat_experiments/models_configs/nnUnet_base_1fold_supervised_less_val/nnUNet_raw/Dataset952_BraTS-GoAT-base-1fold-supervised-less-val/dataset.json')

In [7]:
!nnUNetv2_plan_and_preprocess -d {DATASET_ID} --verify_dataset_integrity -pl {PLANNER} -c {CONFIG} -np 4

Fingerprint extraction...
Dataset952_BraTS-GoAT-base-1fold-supervised-less-val


Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer



####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################



Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer


  0%|                                                  | 0/1351 [00:00<?, ?it/s]

...

100%|██████████████████████████████████████▉| 1348/1351 [22:10<00:02,  1.10it/s]

100%|██████████████████████████████████████▉| 1349/1351 [22:11<00:01,  1.14it/s]

100%|██████████████████████████████████████▉| 1350/1351 [22:12<00:00,  1.35it/s]

100%|███████████████████████████████████████| 1351/1351 [22:14<00:00,  1.32s/it]

100%|███████████████████████████████████████| 1351/1351 [22:14<00:00,  1.01it/s]


In [8]:
preprocessed_dataset = NNUNET_PREP / f"Dataset{DATASET_ID:03d}_{DATASET_NAME}"
plans_files = list(preprocessed_dataset.glob(f"{PLANS_NAME}*.json"))
print("Preprocessed:", preprocessed_dataset, "| existe:", preprocessed_dataset.exists())
print("Plans (L):", [p.name for p in plans_files])

Preprocessed: /home/henrique.grande/brats_goat_experiments/models_configs/nnUnet_base_1fold_supervised_less_val/nnUNet_preprocessed/Dataset952_BraTS-GoAT-base-1fold-supervised-less-val | existe: True
Plans (L): ['nnUNetResEncUNetLPlans.json']


In [9]:
# Split de 1 fold com APENAS 5% para validacao (KFold n_splits=20 -> val = 1/20 = 5%; fold 0). Apenas dados supervisionados.
from sklearn.model_selection import KFold

real_sorted = sorted(train_ids)
kf = KFold(n_splits=20, shuffle=True, random_state=12345)
tr_idx, val_idx = next(iter(kf.split(real_sorted)))   # fold 0 (95% train / 5% val)
tr_real  = [real_sorted[i] for i in tr_idx]
val_real = [real_sorted[i] for i in val_idx]

splits = [{"train": sorted(tr_real), "val": sorted(val_real)}]
with open(preprocessed_dataset / "splits_final.json", "w", encoding="utf-8") as f:
    json.dump(splits, f, indent=2, ensure_ascii=False)

print(f"train={len(splits[0]['train'])} reais | val={len(splits[0]['val'])} reais (~5%)")

train=1283 reais | val=68 reais (~5%)


In [10]:
# Treino supervisionado: 1 fold (fold 0), 1000 epocas (nnUNetTrainer padrao), configuracao Large (ResEnc L)
!nnUNetv2_train {DATASET_ID} {CONFIG} {FOLD} -device cuda -p {PLANS_NAME} -tr {TRAINER}

Using device: cuda:0



#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################



2026-06-21 08:06:17.921443: Using torch.compile...


2026-06-21 08:06:18.743165: do_dummy_2d_data_aug: False
2026-06-21 08:06:18.745132: Using splits from existing split file: /home/henrique.grande/brats_goat_experiments/models_configs/nnUnet_base_1fold_supervised_less_val/nnUNet_preprocessed/Dataset952_BraTS-GoAT-base-1fold-supervised-less-val/splits_final.json
2026-06-21 08:06:18.745285: The split file contains 1 splits.
2026-06-21 08:06:18.745307: Desired fold for training: 0
2026-06-21 08:06:18.745335: This split has 1283 training and 68 validation cases.


using pin_memory on device 0


...

2026-06-22 16:08:52.903912: predicting BraTS-GoAT-02362


2026-06-22 16:08:53.003143: BraTS-GoAT-02362, shape torch.Size([4, 136, 169, 147]), rank 0


2026-06-22 16:08:53.255917: predicting BraTS-GoAT-02483


2026-06-22 16:08:53.334430: BraTS-GoAT-02483, shape torch.Size([4, 139, 178, 142]), rank 0


2026-06-22 16:09:00.439076: Validation complete
2026-06-22 16:09:00.439367: Mean Validation Dice:  0.8128680500876341


In [11]:
# Inferencia na VALIDACAO OFICIAL usando o BEST checkpoint
!nnUNetv2_predict -i "{imagesVs}" -o "{PRED_VAL_OFC}" -d {DATASET_ID} -c {CONFIG} -f {FOLD} -p {PLANS_NAME} -tr {TRAINER} -chk checkpoint_best.pth

n_preds = len(list(PRED_VAL_OFC.glob(f"*{FILE_ENDING}")))
print(f"Validacao oficial: {n_preds} mascaras preditas em\n  {PRED_VAL_OFC}")


#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################



There are 451 cases in the source folder
I am processing 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 451 cases that I would like to predict



Predicting BraTS-GoAT-02489:
perform_everything_on_device: True


  0%|                                                     | 0/1 [00:00<?, ?it/s]

100%|█████████████████████████████████████████████| 1/1 [00:01<00:00,  1.46s/it]


...


Predicting BraTS-GoAT-02939:
perform_everything_on_device: True


  0%|                                                     | 0/1 [00:00<?, ?it/s]

100%|█████████████████████████████████████████████| 1/1 [00:00<00:00,  3.64it/s]


sending off prediction to background worker for resampling and export
done with BraTS-GoAT-02939


Validacao oficial: 451 mascaras preditas em
  /home/henrique.grande/brats_goat_experiments/predictions/nnUnet_base_1fold_supervised_less_val/ofc_validation
